# FlashAttention in depth: the CPU companion

**Tier:** T0. numpy only, no GPU, runs in a minute on a laptop or Colab CPU.

Companion to [`flash-attention-deep-dive.md`](flash-attention-deep-dive.md). Every derived number in that page is
recomputed here, and every algorithm is checked against a naive reference. The formulas live in
[`fa_calculators.py`](fa_calculators.py) (pinned by `test_fa_calculators.py`); this notebook imports them.

**The one-minute version.** After running this notebook you should be able to show, with numbers, that:

1. naive attention is memory-bound at any length because its intensity tends to `d/b`;
2. the online-softmax state `(m, l, o)` merges exactly in any order, which is what tiling, split-KV decode and cascade attention all rely on;
3. an FA2-order tiled forward pass with causal block skipping matches the naive result and visits about half the tiles;
4. the backward pass needs only `O` and the log-sum-exp, not `P`;
5. split-KV decode with GQA packing is exact and reads K/V once per KV head;
6. FP8 errors come from mantissa rounding, from outliers (for integer-like scaling) and from small probabilities flushing to zero, and each has a different fix.

In [ ]:
import math
import numpy as np
import fa_calculators as fc

np.set_printoptions(precision=6, suppress=True)


def check(name, ok):
    assert ok, name
    print(f"PASS  {name}")

## 1. Naive vs tiled attention on the roofline (deep dive, sections 1 and 3.4)

One head, `d = 128`, bf16. Naive traffic is `4N²b + 4Ndb`; the FA2 schedule without any L2 reuse re-reads K and V once
per Q block; the compulsory traffic reads Q, K, V once and writes O once.

In [ ]:
d, b = 128, 2
print(f"{'N':>7} {'naive MB':>9} {'I':>6} {'H100 mem us':>12} {'H100 comp us':>13} {'L4 mem us':>10} "
      f"{'L4 comp us':>11} {'FA2 no-L2 MB':>13} {'compulsory MB':>14}")
for N in (1024, 4096, 32768):
    t = fc.naive_traffic(N, d, b)
    h, l4 = fc.roofline(t["flops"], t["bytes"], "H100"), fc.roofline(t["flops"], t["bytes"], "L4")
    f2 = fc.flash_traffic(N, d, 128, 64, b)
    print(f"{N:>7} {t['bytes']/1e6:>9.1f} {t['intensity']:>6.1f} {h['t_memory_s']*1e6:>12.1f} "
          f"{h['t_compute_s']*1e6:>13.1f} {l4['t_memory_s']*1e6:>10.1f} {l4['t_compute_s']*1e6:>11.1f} "
          f"{f2['bytes']/1e6:>13.1f} {fc.compulsory_bytes(N, d, b)/1e6:>14.2f}")

for dev in ("T4", "L4", "A100", "H100", "B200"):
    print(f"{dev:>5}: ridge {fc.DEVICES[dev].ridge:6.1f} FLOP/B")

check("naive intensity tends to d/b = 64", abs(fc.naive_traffic(1 << 20, d, b)["intensity"] - d / b) < 0.1)
check("FA2 tile intensity is ~B_r FLOP/B in bf16 when K/V come from DRAM",
      abs(fc.flash_traffic(1 << 18, d, 128, 64, b)["intensity"] - 128) < 1.5)

## 2. The softmax state is a monoid (deep dive, section 2)

First the hand-worked example of section 2.6: one query row, `τ = 0.5`, raw scores `[4, 8]` then `[6, 12]`,
values `v1..v4`. Then the same result as a split-KV merge in LSE form, and with an FA4-style lagging max.

In [ ]:
S_blocks = [[4.0, 8.0], [6.0, 12.0]]
V_blocks = [[[1, 0], [0, 1]], [[1, 1], [2, -1]]]
tau = 0.5

ref_o, ref_lse = fc.reference_attention_row(np.array([4, 8, 6, 12.0]),
                                            np.array([[1, 0], [0, 1], [1, 1], [2, -1.0]]), tau)
print("reference  O =", ref_o, "  LSE =", round(ref_lse, 6))
for use_exp2 in (False, True):
    steps, o, lse = fc.online_trace(S_blocks, V_blocks, use_exp2=use_exp2, scale=tau)
    print("\nbase 2 (one FFMA + one EX2 per score)" if use_exp2 else "\nnatural exp")
    for i, s in enumerate(steps, 1):
        print(f"  block {i}: m={s['m']:.0f}  alpha={s['alpha']:.6f}  p={s['p']}  l={s['l']:.6f}  o={s['o_unnorm']}")
    print("  O =", o, "  LSE =", round(lse, 6))
    check("online softmax == reference" + (" (exp2)" if use_exp2 else ""),
          np.allclose(o, ref_o, atol=1e-12) and abs(lse - ref_lse) < 1e-12)

st1 = fc.block_state(tau * np.array(S_blocks[0]), np.array(V_blocks[0], float))
st2 = fc.block_state(tau * np.array(S_blocks[1]), np.array(V_blocks[1], float))
(o1, l1), (o2, l2) = st1.finalize(), st2.finalize()
o, lse = fc.merge_lse(o1, l1, o2, l2)
print(f"\nsplit 1: O={o1} LSE={l1:.6f}   split 2: O={o2} LSE={l2:.6f}")
print(f"merged : O={o} LSE={lse:.6f}   weights {math.exp(l1 - lse):.6f}, {math.exp(l2 - lse):.6f}")
check("LSE-form merge == reference", np.allclose(o, ref_o, atol=1e-12) and abs(lse - ref_lse) < 1e-12)

c = tau * fc.LOG2E
growth = (12 - 8) * c                      # how far the max moved, in log2 units
m_ref = 8.0 if growth < 3.0 else 12.0      # threshold 3 for illustration; FA4 uses 8
p1 = 2 ** (np.array(S_blocks[0]) * c - m_ref * c)
p2 = 2 ** (np.array(S_blocks[1]) * c - m_ref * c)
o_lag = p1 @ np.array(V_blocks[0], float) + p2 @ np.array(V_blocks[1], float)
l_lag = p1.sum() + p2.sum()
print(f"\nmax grew by {growth:.3f} log2 units -> keep m = {m_ref:.0f}; p2 = {p2}; l = {l_lag:.6f}; O = {o_lag / l_lag}")
check("a lagging reference max gives the same O", np.allclose(o_lag / l_lag, ref_o, atol=1e-12))

In [ ]:
rng = np.random.default_rng(0)
keys, vals, q = rng.normal(size=(12, 16)), rng.normal(size=(12, 8)), rng.normal(size=16) * 2
scores = keys @ q / 4.0
pieces = [fc.block_state(scores[i:i + 3], vals[i:i + 3]) for i in range(0, 12, 3)]

seq = fc.empty_state(8)
for p in pieces:                                   # left to right: one CTA's inner loop
    seq = fc.merge(seq, p)
tree = fc.merge(fc.merge(pieces[0], pieces[1]), fc.merge(pieces[2], pieces[3]))   # a reduction tree
rev = fc.empty_state(8)
for p in reversed(pieces):                         # right to left: FA2 walks K/V blocks backwards
    rev = fc.merge(p, rev)
ref, _ = fc.reference_attention_row(scores, vals)
for name, st in (("sequential", seq), ("tree", tree), ("reversed", rev)):
    check(f"{name} merge == reference", np.allclose(st.finalize()[0], ref, atol=1e-12))
empty = fc.merge(fc.empty_state(8), fc.empty_state(8))
check("merging two empty states: l = 0, no NaN (the -inf guard)", empty.l == 0.0 and not np.isnan(empty.o).any())

## 3. An FA2-order tiled forward pass with causal skipping (deep dive, sections 3 and 4)

One "CTA" per Q block (the outer loop is the grid), K/V blocks inner, walked from the last block down so that the
masked diagonal tiles come first. Ragged lengths, `B_r ≠ B_c`, and rows that are fully masked inside the first tile
they visit are all exercised. The tile counts must match `fa_calculators.causal_tiles()`.

In [ ]:
def naive(Q, K, V, causal=False):
    S = (Q @ K.T) / math.sqrt(Q.shape[1])
    if causal:
        S = np.where(np.tril(np.ones(S.shape, dtype=bool)), S, -np.inf)
    mx = S.max(axis=1, keepdims=True)
    P = np.exp(S - mx)
    return (P / P.sum(axis=1, keepdims=True)) @ V, (mx[:, 0] + np.log(P.sum(axis=1)))


def fa2_forward(Q, K, V, Br, Bc, causal=False):
    N, d = Q.shape
    tau = 1.0 / math.sqrt(d)
    O, L = np.zeros_like(Q), np.zeros(N)
    visited = masked = 0
    for m in range(math.ceil(N / Br)):                               # grid axis: one CTA per Q block
        r0, r1 = m * Br, min((m + 1) * Br, N)
        mi, li, acc = np.full(r1 - r0, -np.inf), np.zeros(r1 - r0), np.zeros((r1 - r0, V.shape[1]))
        n_max = math.ceil(N / Bc)
        if causal:
            n_max = min(n_max, math.ceil((m + 1) * Br / Bc))          # stop at the diagonal
        for n in reversed(range(n_max)):                             # diagonal (masked) tiles first
            c0, c1 = n * Bc, min((n + 1) * Bc, N)
            S = (Q[r0:r1] @ K[c0:c1].T) * tau
            visited += 1
            if causal and c1 - 1 > r0:                               # tile crosses the diagonal
                masked += 1
                S = np.where(np.arange(c0, c1)[None, :] <= np.arange(r0, r1)[:, None], S, -np.inf)
            m_new = np.maximum(mi, S.max(axis=1))
            m_safe = np.where(np.isneginf(m_new), 0.0, m_new)         # guard: no (-inf) - (-inf)
            alpha = np.exp(np.where(np.isneginf(mi), -np.inf, mi - m_safe))
            P = np.exp(S - m_safe[:, None])
            li = alpha * li + P.sum(axis=1)
            acc = alpha[:, None] * acc + P @ V[c0:c1]
            mi = m_new
        O[r0:r1] = acc / li[:, None]                                 # normalize once
        L[r0:r1] = mi + np.log(li)                                   # natural-log LSE of scaled scores
    return O, L, visited, masked


rng = np.random.default_rng(1)
for N, Br, Bc in ((300, 64, 32), (512, 128, 64), (257, 32, 64)):
    Q, K, V = (rng.normal(size=(N, 64)) for _ in range(3))
    for causal in (False, True):
        O, L, visited, masked = fa2_forward(Q, K, V, Br, Bc, causal)
        O_ref, L_ref = naive(Q, K, V, causal)
        expected = fc.causal_tiles(N, N, Br, Bc)[0] if causal else math.ceil(N / Br) * math.ceil(N / Bc)
        full = math.ceil(N / Br) * math.ceil(N / Bc)
        check(f"N={N} Br={Br} Bc={Bc} causal={causal!s:5}: O, LSE match; {visited}/{full} tiles visited",
              np.allclose(O, O_ref, atol=1e-10) and np.allclose(L, L_ref, atol=1e-10) and visited == expected)

v, mk, full = fc.causal_tiles(512, 512, 128, 64)
print(f"\nthe diagram in section 4.4: N=512, 128 x 64 tiles -> {v} of {full} visited, {mk} masked")
for N in (4096, 32768):
    v, mk, full = fc.causal_tiles(N, N, 128, 64)
    print(f"N={N:>6}: {v} of {full} tiles ({v/full:.1%}), {mk} masked")

## 4. The backward pass from `O` and the log-sum-exp (deep dive, section 3.5)

FA2's order: one "CTA" per K/V block, Q blocks inner, `P` recomputed as `exp(S − L)`, `D = rowsum(dO ∘ O)` from a
pre-pass, and `dQ` accumulated across CTAs (on the GPU with `atomicAdd`). Checked against the dense analytic gradient
and a finite difference.

In [ ]:
def dense_grads(Q, K, V, dO, causal):
    tau = 1 / math.sqrt(Q.shape[1])
    S = (Q @ K.T) * tau
    if causal:
        S = np.where(np.tril(np.ones(S.shape, dtype=bool)), S, -np.inf)
    P = np.exp(S - S.max(1, keepdims=True))
    P /= P.sum(1, keepdims=True)
    dP = dO @ V.T
    dS = P * (dP - (dP * P).sum(1, keepdims=True))
    return tau * dS @ K, tau * dS.T @ Q, P.T @ dO


def fa_backward(Q, K, V, O, L, dO, Br, Bc, causal):
    N, d = Q.shape
    tau = 1 / math.sqrt(d)
    D = (dO * O).sum(axis=1)                                   # the O(Nd) pre-pass
    dQ, dK, dV = np.zeros_like(Q), np.zeros_like(K), np.zeros_like(V)
    for n in range(math.ceil(N / Bc)):                         # one CTA per K/V block
        c0, c1 = n * Bc, min((n + 1) * Bc, N)
        for m in range(math.ceil(N / Br)):
            r0, r1 = m * Br, min((m + 1) * Br, N)
            if causal and c0 > r1 - 1:
                continue                                        # tile entirely above the diagonal
            S = (Q[r0:r1] @ K[c0:c1].T) * tau
            if causal:
                S = np.where(np.arange(c0, c1)[None, :] <= np.arange(r0, r1)[:, None], S, -np.inf)
            P = np.exp(S - L[r0:r1, None])                      # recomputed: no max, no sum
            dV[c0:c1] += P.T @ dO[r0:r1]
            dS = P * (dO[r0:r1] @ V[c0:c1].T - D[r0:r1, None])
            dQ[r0:r1] += tau * dS @ K[c0:c1]                    # atomicAdd on the GPU
            dK[c0:c1] += tau * dS.T @ Q[r0:r1]
    return dQ, dK, dV


rng = np.random.default_rng(2)
N, d = 200, 32
Q, K, V, dO = (rng.normal(size=(N, d)) for _ in range(4))
for causal in (False, True):
    O, L, _, _ = fa2_forward(Q, K, V, 64, 32, causal)
    got, want = fa_backward(Q, K, V, O, L, dO, 64, 32, causal), dense_grads(Q, K, V, dO, causal)
    check(f"tiled backward == dense gradients (causal={causal})",
          all(np.allclose(g_, w_, atol=1e-10) for g_, w_ in zip(got, want)))

eps, (i, j) = 1e-6, (5, 3)
loss = lambda Qx: float((naive(Qx, K, V, True)[0] * dO).sum())
Qp, Qm = Q.copy(), Q.copy()
Qp[i, j] += eps
Qm[i, j] -= eps
fd = (loss(Qp) - loss(Qm)) / (2 * eps)
O, L, _, _ = fa2_forward(Q, K, V, 64, 32, True)
check(f"finite difference dQ[{i},{j}] = {fd:.6f} matches", abs(fd - fa_backward(Q, K, V, O, L, dO, 64, 32, True)[0][i, j]) < 1e-5)

print(f"\nsaved for backward at N=32k, 32 heads: LSE {32*32768*4/1e6:.1f} MB vs P in bf16 {32*32768**2*2/1e9:.1f} GB")
print(f"backward / forward FLOPs: {fc.attention_flops(4096, 4096, 128, pass_='bwd') / fc.attention_flops(4096, 4096, 128):.1f}"
      " (5 GEMMs vs 2)")

## 5. Split-KV decode with GQA packing (deep dive, section 6)

The `g = 4` query heads that share one KV head form a 4-row tile. Each split (one CTA on the GPU) reads its slice of
K/V once for all 4 rows and writes `(Ô, LSE)`; a combine step merges them (section 2.3). Then the split counts FA2's
heuristic would choose, and how decode time grows with batch × context.

In [ ]:
rng = np.random.default_rng(7)
g, L_ctx, d = 4, 5000, 128
q = rng.normal(size=(g, d))
Kc, Vc = rng.normal(size=(L_ctx, d)), rng.normal(size=(L_ctx, d))
tau = 1 / math.sqrt(d)


def split_kv_packed(q, K, V, n_splits):
    bounds = np.linspace(0, K.shape[0], n_splits + 1).astype(int)
    outs, lses = [], []
    for lo, hi in zip(bounds[:-1], bounds[1:]):       # one CTA per split
        S = (q @ K[lo:hi].T) * tau                     # a g-row tile: K/V read once for all g heads
        m = S.max(axis=1)
        P = np.exp(S - m[:, None])
        l = P.sum(axis=1)
        outs.append((P @ V[lo:hi]) / l[:, None])
        lses.append(m + np.log(l))
    outs, lses = np.stack(outs), np.stack(lses)       # fp32 partials in HBM on the GPU
    top = lses.max(axis=0)
    lse = top + np.log(np.exp(lses - top).sum(axis=0))  # combine kernel
    return (np.exp(lses - lse)[:, :, None] * outs).sum(axis=0), lse


S_ref = (q @ Kc.T) * tau
P_ref = np.exp(S_ref - S_ref.max(1, keepdims=True))
O_ref = (P_ref / P_ref.sum(1, keepdims=True)) @ Vc
for n_splits in (1, 3, 29, 128):
    check(f"split-KV decode with {n_splits:3d} splits == reference",
          np.allclose(split_kv_packed(q, Kc, Vc, n_splits)[0], O_ref, atol=1e-12))

print(f"\nK/V bytes per KV head (bf16): packed {2*L_ctx*d*2/1e6:.2f} MB; one query head at a time {g*2*L_ctx*d*2/1e6:.2f} MB")
print(f"decode intensity: MHA {fc.decode_intensity(1):.0f}, GQA g=4 {fc.decode_intensity(4):.0f}, "
      f"MLA absorbed {fc.mla_decode_intensity():.0f} (FP8 latent {fc.mla_decode_intensity(b=1):.0f}) FLOP/B")

print(f"\n{'batch x ctx':>14} {'GPU':>5} {'CTAs':>5} {'splits':>7} {'launched':>9}")
for B, ctx in ((1, 32768), (8, 32768), (64, 4096), (4, 131072)):
    for dev in ("H100", "A100", "L4"):
        r = fc.fa2_decode_splits(B, 32, 8, ctx, 128, fc.DEVICES[dev].sms)
        print(f"{B:>4} x {ctx:>7} {dev:>5} {r['ctas_without_split']:>5} {r['splits']:>7} {r['ctas']:>9}")
check("H100, batch 1 x 32k: 8 CTAs -> 29 splits", fc.fa2_decode_splits(1, 32, 8, 32768, 128, 132)["splits"] == 29)

w_bytes, bw = 8.03e9 * 2, fc.DEVICES["H100"].hbm_tbs * 1e12       # Llama-3-8B shape, bf16, H100
print(f"\nweights alone: {w_bytes/bw*1e3:.2f} ms per step")
for B, ctx in ((1, 2048), (1, 32768), (1, 131072), (32, 2048), (32, 8192), (8, 32768)):
    kv = fc.decode_attention_bytes(B * ctx, 32, 8, 128)
    print(f"{B:>3} x {ctx:>6}: KV {kv/1e9:6.2f} GB, {kv/bw*1e3:6.2f} ms, attention share of step bytes {kv/(kv+w_bytes):6.1%}")

## 6. FP8 numerics (deep dive, sections 5.6 and 9.4)

Three separate error sources, three separate fixes:

1. **Mantissa rounding.** e4m3 keeps 3 mantissa bits, so every element carries up to 6.25% relative error. No scaling
   or rotation fixes that; only a wider format does.
2. **Outliers.** A per-tensor scale is set by the largest value. For a floating-point format that barely matters (relative
   precision does not depend on magnitude); for integer formats (INT8/INT4 KV caches) the step size is `amax/127`, so an
   outlier channel costs every other value its precision. A random Hadamard rotation spreads the outlier over all
   channels and leaves `QKᵀ` unchanged.
3. **Small probabilities.** `P ≤ 1`, and e4m3's smallest subnormal is `2^-9`. FA3 multiplies `P` by `2^8` before the
   conversion (`Max_offset = 8`) and divides the row sum by the same factor.

In [ ]:
rng = np.random.default_rng(0)
N, d = 256, 128
q, k = rng.normal(size=(N, d)), rng.normal(size=(N, d))
k[:, 7] *= 20.0                                             # one outlier channel in K
ref = q @ k.T
Mrot = fc.random_hadamard(d, seed=0)
check("the rotation leaves QK^T unchanged", np.allclose((q @ Mrot) @ (k @ Mrot).T, ref))


def int_quant(bits):
    top = 2 ** (bits - 1) - 1
    return lambda x: np.round(x / (np.abs(x).max() / top)) * (np.abs(x).max() / top)


def rel_err(quant, qq, kk):
    return np.linalg.norm(quant(qq) @ quant(kk).T - ref) / np.linalg.norm(ref)


print(f"amax/rms of K: plain {np.abs(k).max() / np.sqrt((k**2).mean()):.1f}, "
      f"rotated {np.abs(k @ Mrot).max() / np.sqrt(((k @ Mrot)**2).mean()):.1f}")
res = {}
for name, quant in (("fp8 e4m3", lambda x: fc.quantize_fp8(x)[0]), ("int8", int_quant(8)), ("int4", int_quant(4))):
    res[name] = (rel_err(quant, q, k), rel_err(quant, q @ Mrot, k @ Mrot))
    print(f"{name:>9}, per-tensor scale: QK^T relative error {res[name][0]:6.2%} plain, {res[name][1]:6.2%} rotated")
check("rotation cuts the int8 error by more than 3x", res["int8"][1] < res["int8"][0] / 3)
check("rotation barely moves the fp8 error (it is mantissa rounding)", abs(res["fp8 e4m3"][1] - res["fp8 e4m3"][0]) < 0.01)

rng = np.random.default_rng(1)
s = rng.normal(size=4096) * 2.0                             # one row of scaled scores
p = np.exp(s - s.max())
v = rng.normal(size=(4096, 64))
o_ref = p @ v / p.sum()
flushed = {}
for offset in (0, 8):
    pq = fc.round_to_e4m3(p * 2.0**offset) / 2.0**offset   # P as it enters the P.V MMA
    o = pq @ v / p.sum()                                    # the row sum l is accumulated in fp32 from unrounded p
    flushed[offset] = np.mean(pq == 0)
    print(f"P in e4m3, offset 2^{offset}: {flushed[offset]:6.1%} of probabilities flush to zero, "
          f"probability mass lost {1 - pq.sum() / p.sum():+.3%}, output error {np.linalg.norm(o - o_ref) / np.linalg.norm(o_ref):.2%}")
check("the 2^8 offset keeps small probabilities out of the flush-to-zero range", flushed[8] < flushed[0] / 10)

## In a design review

**Q1. Someone proposes making naive attention faster by increasing the batch size. Why won't that help?**
Batch, heads and sequence length scale FLOPs and bytes together; the naive schedule's intensity tends to `d/b`
(64 FLOP/B at `d = 128`, bf16), below every GPU's ridge. Only a schedule change (fusion and tiling) moves it.

**Q2. Split-KV decode writes partial results to HBM and adds a kernel. When is that worth it?**
When `batch × KV heads` CTAs cannot fill the SMs (a few sequences, long contexts). The partials are tiny next to the
K/V bytes (0.7% at batch 1 × 32k on an H100), and the extra CTAs are what saturate bandwidth. At large batch the
heuristic picks one split.

**Q3. Would a Hadamard rotation fix our FP8 attention accuracy problem?**
Only if the problem is outliers under integer-like scaling (INT8/INT4 KV caches, coarse block scales that push values
into subnormals). For e4m3 with per-tensor scales the dominant error is the 3-bit mantissa, which the rotation does not
touch; measure end-to-end quality before and after.